In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [3]:
#sc.stop() # to ensure that this is the context being executed
credentials_location = '/Users/mafalda.rodrigues/.gc/keys/my-creds.json'


conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar,./lib/spark-bigquery-with-dependencies_2.12-0.24.0.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [4]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

25/04/18 17:29:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 17:29:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/18 17:29:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [5]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [6]:
df = spark.read.csv('gs://public-life-data-seattle-bucket/*.csv', header=True, inferSchema=True)


In [7]:
df.count()

17281

In [10]:
df = spark.read \
    .format("bigquery") \
    .option("table", "project-zoomcamp-457121.public_life_data_seattle_dataset.public_life_data_seattle") \
    .load()

df.show()

Py4JJavaError: An error occurred while calling o140.load.
: java.lang.RuntimeException: Table project-zoomcamp-457121.public_life_data_seattle_dataset.public_life_data_seattle not found
	at scala.sys.package$.error(package.scala:30)
	at com.google.cloud.spark.bigquery.BigQueryRelationProvider.$anonfun$createRelationInternal$1(BigQueryRelationProvider.scala:80)
	at scala.Option.getOrElse(Option.scala:189)
	at com.google.cloud.spark.bigquery.BigQueryRelationProvider.createRelationInternal(BigQueryRelationProvider.scala:80)
	at com.google.cloud.spark.bigquery.BigQueryRelationProvider.createRelation(BigQueryRelationProvider.scala:46)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:346)
	at org.apache.spark.sql.DataFrameReader.loadV1Source(DataFrameReader.scala:229)
	at org.apache.spark.sql.DataFrameReader.$anonfun$load$2(DataFrameReader.scala:211)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameR

In [9]:
# Escrevendo no BigQuery na tabela public_life_data_seattle

project_id = 'project-zoomcamp-457121'
dataset_id = 'public_life_data_seattle_dataset'
table_id = 'public_life_data_seattle'


df.write \
    .format('bigquery') \
    .option('temporaryGcsBucket', 'public-life-data-seattle-bucket') \
    .option('table', f'{project_id}:{dataset_id}.{table_id}') \
    .mode("overwrite") \
    .save()


Py4JJavaError: An error occurred while calling o135.save.
: java.lang.RuntimeException: Failed to write to BigQuery
	at com.google.cloud.spark.bigquery.BigQueryWriteHelper.writeDataFrameToBigQuery(BigQueryWriteHelper.scala:69)
	at com.google.cloud.spark.bigquery.BigQueryInsertableRelation.insert(BigQueryInsertableRelation.scala:43)
	at com.google.cloud.spark.bigquery.BigQueryRelationProvider.createRelation(BigQueryRelationProvider.scala:112)
	at org.apache.spark.sql.execution.datasources.SaveIntoDataSourceCommand.run(SaveIntoDataSourceCommand.scala:48)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult$lzycompute(commands.scala:75)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResult(commands.scala:73)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.executeCollect(commands.scala:84)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyExecuteCommands$1.$anonfun$applyOrElse$1(QueryExecution.scala:107)
	a